# 한국어 비속어/욕설 감지 예제

이 노트북은 캐글(Kaggle Notebook) 환경에서 Hugging Face 모델 `2tle/korean-curse-detection`을 사용하여 한국어 문장의 비속어/욕설 여부를 감지하는 예제입니다.

> 캐글에서 Hugging Face 모델 다운로드가 되지 않는 경우, 오른쪽 설정 패널에서 **Internet 옵션을 On**으로 켜 주세요.

모델 예측은 100% 정확하지 않을 수 있으므로, 실제 서비스에서는 사전 기반 필터, 사용자 신고, 후처리 정책과 함께 사용하는 것이 좋습니다.

## 1단계: 패키지 설치

캐글 환경에 이미 설치된 패키지도 있지만, 실행 안정성을 위해 필요한 패키지를 설치합니다.

In [ ]:
# 캐글 노트북에서 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 빠르게 넘어갈 수 있습니다.
!pip install -q transformers accelerate pandas torch

## 2단계: 라이브러리 불러오기

`transformers.pipeline`을 사용해 텍스트 분류 모델을 간단하게 실행합니다.

In [ ]:
import pandas as pd
import torch
from transformers import pipeline

# 모델 라벨을 사람이 읽기 쉬운 한글 설명으로 바꿔 주는 딕셔너리입니다.
LABEL_MAP = {
    "LABEL_0": "정상 문장",
    "LABEL_1": "비속어/욕설 의심",
}

# GPU가 있으면 GPU를 사용하고, 없으면 CPU로 실행합니다.
# 캐글 무료 CPU 환경에서도 실행 가능하도록 구성했습니다.
device = 0 if torch.cuda.is_available() else -1

print("사용 장치:", "GPU" if device == 0 else "CPU")

## 3단계: 모델 로드

Hugging Face 모델 `2tle/korean-curse-detection`을 로드합니다.

모델 다운로드가 실패한다면 캐글 노트북의 **Internet 옵션이 On**인지 확인해 주세요.

In [ ]:
MODEL_NAME = "2tle/korean-curse-detection"

# transformers의 pipeline을 사용해 텍스트 분류 파이프라인을 생성합니다.
curse_classifier = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device,
)

print("모델 로드 완료:", MODEL_NAME)

## 4단계: 단일 문장 테스트

문자열 하나를 넣어 비속어/욕설 여부를 확인합니다.

In [ ]:
# 단일 문장 테스트 예시입니다.
single_text = "오늘 날씨가 정말 좋네요."

single_prediction = curse_classifier(single_text)[0]
single_label = single_prediction["label"]
single_result = LABEL_MAP.get(single_label, single_label)
single_score = round(single_prediction["score"], 4)

print("입력 문장:", single_text)
print("예측 라벨:", single_label)
print("욕설 여부:", single_result)
print("신뢰도 점수:", single_score)

## 5단계: 여러 문장 일괄 테스트

여러 문장을 리스트로 넣고 한 번에 검사합니다. 아래 리스트에 사용자가 직접 문장을 추가해서 테스트할 수 있습니다.

테스트 목적상 일부 비속어 변형 예시를 최소한으로 포함했습니다.

In [ ]:
# 사용자가 직접 문장을 추가하거나 수정해서 테스트할 수 있습니다.
test_texts = [
    "오늘 수업 정말 재미있었다.",
    "이 영화는 조금 지루했지만 볼 만했어.",
    "친절하게 설명해 주셔서 감사합니다.",
    "ㅅㅂ 이게 왜 안 되지",
    "시1발 진짜 너무하네",
    "씨발 이건 아니지",
    "개짜증나 정말",
    "너무 화가 나지만 차분히 다시 해볼게.",
    # 여기에 직접 테스트할 문장을 추가해 보세요.
    # "테스트할 문장을 여기에 입력하세요.",
]

# 리스트에 담긴 여러 문장을 한 번에 모델에 입력합니다.
batch_predictions = curse_classifier(test_texts)

batch_predictions[:3]

## 6단계: 결과 DataFrame 출력

결과를 보기 쉽게 `pandas.DataFrame`으로 정리합니다.

In [ ]:
rows = []

for text, prediction in zip(test_texts, batch_predictions):
    label = prediction["label"]
    rows.append({
        "입력 문장": text,
        "예측 라벨": label,
        "욕설 여부": LABEL_MAP.get(label, label),
        "신뢰도 점수": round(prediction["score"], 4),
    })

result_df = pd.DataFrame(rows)
result_df

## 7단계: 재사용 가능한 함수 만들기

`detect_curse(text)` 함수는 문자열 하나를 입력받아 결과 딕셔너리를 반환합니다.

In [ ]:
def detect_curse(text):
    """한국어 문장 하나의 비속어/욕설 의심 여부를 검사합니다."""
    prediction = curse_classifier(text)[0]
    label = prediction["label"]

    return {
        "text": text,
        "label": label,
        "result": LABEL_MAP.get(label, label),
        "score": round(prediction["score"], 4),
    }


# 함수 반환 예시입니다.
detect_curse("입력 문장")

In [ ]:
# 비속어/욕설 의심 문장 테스트 예시입니다.
detect_curse("ㅅㅂ 이게 뭐야")

## 8단계: 실제 서비스 적용 시 주의사항

- 모델 예측은 100% 정확하지 않습니다.
- 신조어, 초성, 띄어쓰기 변형, 숫자 치환 표현은 모델이 놓치거나 과하게 탐지할 수 있습니다.
- 실제 서비스에서는 모델만 단독으로 사용하기보다 사전 필터, 정규식 기반 필터, 사용자 신고, 운영자 검수 정책과 함께 사용하는 것이 좋습니다.
- 욕설 탐지는 맥락에 영향을 받습니다. 예를 들어 인용문, 교육 목적의 문장, 신고 처리 화면에서는 별도 정책이 필요할 수 있습니다.
- 캐글에서 모델 다운로드가 되지 않으면 노트북 설정의 Internet 옵션을 On으로 켰는지 확인해야 합니다.